# 缓存与锁 —— 从原理到实战

本笔记围绕项目中 `app/core/lock.py` 和 `app/core/cache.py` 的实现，帮助初学者理解以下概念：

1. **为什么需要锁？** —— 并发场景下的数据竞争问题
2. **读写锁（RWLock）** —— 读多写少场景的性能优化
3. **按 key 粒度的锁（KeyLock）** —— 细粒度并发控制
4. **缓存后端抽象（Protocol）** —— 面向接口编程
5. **内存缓存（MemoryCache）** —— 结合读写锁的本地缓存
6. **工厂模式 + 依赖注入** —— 灵活切换缓存后端
7. **Double-Check Locking 实战** —— 项目中的真实用法

---
## 一、为什么需要锁？

In [ ]:
import threading

# 一个没有锁保护的共享计数器
counter = 0

def increment(n: int):
    global counter
    for _ in range(n):
        counter += 1  # 这不是原子操作！读取 → 加1 → 写回，中间可能被打断

# 启动 10 个线程，每个 +100000
threads = [threading.Thread(target=increment, args=(100_000,)) for _ in range(10)]
for t in threads:
    t.start()
for t in threads:
    t.join()

print(f"期望结果: {10 * 100_000}")
print(f"实际结果: {counter}")
print(f"数据丢失: {10 * 100_000 - counter} 次更新被覆盖")

### 发生了什么？

`counter += 1` 看起来是一行代码，实际上包含 3 步：
1. **读取** counter 的当前值
2. **计算** 当前值 + 1
3. **写回** 新值

线程 A 读到 `counter=100`，还没来得及写回，线程 B 也读到了 `counter=100`。
两个线程都写回 `101`，但实际上应该变成 `102`。这就是**数据竞争（Race Condition）**。

解决方案：用 `threading.Lock` 把 "读-改-写" 包成原子操作。

In [ ]:
counter = 0
lock = threading.Lock()

def safe_increment(n: int):
    global counter
    for _ in range(n):
        with lock:      # 加锁：同一时刻只有一个线程能进入
            counter += 1

threads = [threading.Thread(target=safe_increment, args=(100_000,)) for _ in range(10)]
for t in threads:
    t.start()
for t in threads:
    t.join()

print(f"期望结果: {10 * 100_000}")
print(f"实际结果: {counter}")
print("锁保护下，结果正确！")

---
## 二、读写锁（RWLock）—— 读多写少场景的优化

普通的 `threading.Lock` 的问题是：**读操作之间也会互斥**。

但缓存场景是典型的 **"读多写少"**：
- 多个线程同时读缓存 → 完全安全，不应该互相等待
- 只有线程在写缓存时 → 才需要独占

这就是 **读写锁（Read-Write Lock）** 的设计动机：
| 场景 | 是否允许并发 |
|------|------------|
| 读 + 读 | 允许 |
| 读 + 写 | 不允许 |
| 写 + 写 | 不允许 |

下面是项目中 `RWLock` 的实现（简化注释版）：

In [ ]:
import threading
from contextlib import contextmanager
from typing import Generator


class RWLock:
    """读写锁：多个读者可并发持有，写者独占。写者优先策略防止写者饥饿。"""

    def __init__(self) -> None:
        self._lock = threading.Lock()               # 底层互斥锁，保护内部状态
        self._readers_ok = threading.Condition(self._lock)  # 读者等待条件
        self._writers_ok = threading.Condition(self._lock)  # 写者等待条件
        self._active_readers = 0   # 当前正在读的线程数
        self._active_writers = 0   # 当前正在写的线程数（最多1）
        self._waiting_writers = 0  # 正在排队等待写的线程数

    @contextmanager
    def read(self) -> Generator[None, None, None]:
        """获取读锁。"""
        with self._lock:
            # 如果有活跃写者 或 有等待中的写者 → 读者等待（写者优先策略）
            while self._active_writers > 0 or self._waiting_writers > 0:
                self._readers_ok.wait()
            self._active_readers += 1
        try:
            yield  # ---- 临界区：可以安全地读 ----
        finally:
            with self._lock:
                self._active_readers -= 1
                if self._active_readers == 0:
                    self._writers_ok.notify()  # 唤醒一个等待的写者

    @contextmanager
    def write(self) -> Generator[None, None, None]:
        """获取写锁。"""
        with self._lock:
            self._waiting_writers += 1  # 先登记为 "等待中"
            while self._active_readers > 0 or self._active_writers > 0:
                self._writers_ok.wait()  # 等待所有读者和写者退出
            self._waiting_writers -= 1   # 从等待队列移出
            self._active_writers += 1    # 标记为活跃写者
        try:
            yield  # ---- 临界区：独占写入 ----
        finally:
            with self._lock:
                self._active_writers -= 1
                self._readers_ok.notify_all()  # 唤醒所有等待的读者
                self._writers_ok.notify()       # 也唤醒一个等待的写者

### 关键设计点

1. **`threading.Condition`** = `Lock` + `wait/notify` 机制
   - `wait()`：释放锁并阻塞，直到被唤醒后重新获取锁
   - `notify()` / `notify_all()`：唤醒一个/全部等待线程

2. **写者优先策略**：读者在 `_waiting_writers > 0` 时也会等待。
   - 如果不加这个条件，源源不断的读者可能让写者永远等下去（**写者饥饿**）
   - 一旦有写者在等，新读者就排队，保证写者能及时执行

3. **使用 `@contextmanager`**：让锁可以用 `with` 语句管理，确保异常时也能释放

In [ ]:
import time

rwlock = RWLock()
shared_data = {"value": 0}
log = []

def reader(name: str, delay: float):
    with rwlock.read():
        log.append(f"{time.strftime('%H:%M:%S')} {name} 开始读，value={shared_data['value']}")
        time.sleep(delay)  # 模拟读操作耗时
        log.append(f"{time.strftime('%H:%M:%S')} {name} 读结束")

def writer(name: str, new_value: int, delay: float):
    with rwlock.write():
        log.append(f"{time.strftime('%H:%M:%S')} {name} 开始写")
        shared_data["value"] = new_value
        time.sleep(delay)  # 模拟写操作耗时
        log.append(f"{time.strftime('%H:%M:%S')} {name} 写结束，value={shared_data['value']}")

# 演示：读 + 读并发，写独占
threads = [
    threading.Thread(target=reader,  args=("Reader-A", 0.3)),
    threading.Thread(target=reader,  args=("Reader-B", 0.3)),  # 应该和 A 并发执行
    threading.Thread(target=writer,  args=("Writer-C", 100, 0.3)),  # 应该等 A、B 都结束
    threading.Thread(target=reader,  args=("Reader-D", 0.1)),  # 应该等 C 结束
]
for t in threads:
    t.start()
for t in threads:
    t.join()

for line in log:
    print(line)

print(f"\n最终值: {shared_data['value']}")

观察输出日志的时间戳：
- Reader-A 和 Reader-B 的 "开始读" 几乎同时出现 → **读读并发**
- Writer-C 必须等两个 Reader 都结束才开始写 → **读写互斥**
- Reader-D 必须等 Writer-C 结束才能开始读 → **写读互斥**

---
## 三、按 key 粒度的锁（KeyLock）

假设你在下载不同 ID 的漫画，ID=1 和 ID=2 是完全独立的任务。
如果用一把全局锁，下载 ID=1 时 ID=2 也得等着，这不合理。

**KeyLock** 的思路：每个 key 分配一把独立的锁，不同 key 互不阻塞。

```
KeyLock
├── key=1 → Lock A   ← 线程X持有，线程Y等待
├── key=2 → Lock B   ← 线程Z持有（不受 key=1 影响）
└── key=3 → （尚未创建，首次使用时自动创建）
```

In [ ]:
class KeyLock:
    """按 key 粒度分配独立锁，不同 key 互不阻塞。"""

    def __init__(self) -> None:
        self._guard = threading.Lock()              # 保护 _locks 字典本身
        self._locks: dict[int, threading.Lock] = {} # key → Lock 的映射

    @contextmanager
    def acquire(self, key: int) -> Generator[None, None, None]:
        with self._guard:
            # 首次访问某 key 时，自动创建对应的锁
            if key not in self._locks:
                self._locks[key] = threading.Lock()
            lock = self._locks[key]
        # 注意：在 _guard 释放后再获取 key 对应的锁，避免持有多把锁导致死锁
        with lock:
            yield

In [ ]:
key_lock = KeyLock()
log2 = []

def task(name: str, key: int, duration: float):
    log2.append(f"{time.strftime('%H:%M:%S')} {name} 等待 key={key}")
    with key_lock.acquire(key):
        log2.append(f"{time.strftime('%H:%M:%S')} {name} 获取锁 key={key}，开始工作")
        time.sleep(duration)
        log2.append(f"{time.strftime('%H:%M:%S')} {name} 释放锁 key={key}")

# 任务 A、B 使用 key=1（互斥）；任务 C 使用 key=2（不受影响）
threads = [
    threading.Thread(target=task, args=("Task-A", 1, 0.5)),
    threading.Thread(target=task, args=("Task-B", 1, 0.3)),  # 同 key，等 A
    threading.Thread(target=task, args=("Task-C", 2, 0.3)),  # 不同 key，并发
]
for t in threads:
    t.start()
for t in threads:
    t.join()

for line in log2:
    print(line)

观察输出：
- Task-A (key=1) 和 Task-C (key=2) 应该几乎同时开始 → **不同 key 并发**
- Task-B (key=1) 必须等 Task-A 结束才能开始 → **相同 key 互斥**

---
## 四、缓存后端抽象（Protocol）

项目使用了 Python 的 `typing.Protocol` 定义缓存接口，这是一种 **面向接口编程** 的思路：

| 概念 | 说明 |
|------|------|
| `Protocol` | Python 的结构化子类型（鸭子类型的类型检查版） |
| `@runtime_checkable` | 允许在运行时用 `isinstance()` 检查 |

```python
@runtime_checkable
class CacheBackend(Protocol):
    def get(self, key: str) -> str | None: ...
    def set(self, key: str, value: str, ex: int | None = None) -> ...
    def delete(self, key: str) -> None: ...
    def exists(self, key: str) -> bool: ...
```

**好处**：只要一个类实现了这 4 个方法，它就自动成为 `CacheBackend`，不需要显式继承。
这让 `MemoryCache`（本地测试）和 `RedisCache`（生产环境）可以无缝互换。

In [ ]:
from typing import Protocol, runtime_checkable

@runtime_checkable
class Animal(Protocol):
    def speak(self) -> str: ...

class Dog:
    def speak(self) -> str:
        return "汪汪！"

class Cat:
    def speak(self) -> str:
        return "喵喵！"

class Rock:
    pass  # 没有实现 speak

# 不需要 class Dog(Animal)，只要有 speak 方法就算 Animal
print(f"Dog 是 Animal: {isinstance(Dog(), Animal)}")
print(f"Cat 是 Animal: {isinstance(Cat(), Animal)}")
print(f"Rock 是 Animal: {isinstance(Rock(), Animal)}")

# 用 Protocol 做类型标注，函数可以接受任何有 speak 方法的对象
def make_sound(animal: Animal):
    print(animal.speak())

make_sound(Dog())
make_sound(Cat())

---
## 五、内存缓存（MemoryCache）—— 结合读写锁的本地缓存

项目中的 `MemoryCache` 使用字典作为存储，配合 `RWLock` 保证线程安全。

核心数据结构：`dict[str, tuple[str, float | None]]`
- key → (value, expire_at)
- `expire_at` 为 `None` 表示永不过期
- `expire_at` 为时间戳，读取时检查是否已过期

In [ ]:
import time

class MemoryCache:
    def __init__(self) -> None:
        self._store: dict[str, tuple[str, float | None]] = {}
        self._rwlock = RWLock()  # 使用上面实现的读写锁

    def get(self, key: str) -> str | None:
        with self._rwlock.read():               # 读锁：多个线程可以同时读
            if key not in self._store:
                return None
            value, expire_at = self._store[key]
            if expire_at and time.time() > expire_at:
                return None                      # 已过期，返回 None（惰性删除）
            return value

    def set(self, key: str, value: str, ex: int | None = None) -> None:
        with self._rwlock.write():               # 写锁：独占写入
            expire_at = time.time() + ex if ex else None
            self._store[key] = (value, expire_at)

    def delete(self, key: str) -> None:
        with self._rwlock.write():               # 写锁：独占删除
            self._store.pop(key, None)

    def exists(self, key: str) -> bool:
        return self.get(key) is not None

# 演示
cache = MemoryCache()
cache.set("user:1", "Alice")
cache.set("session:abc", "token123", ex=2)  # 2 秒后过期

print(f"user:1 → {cache.get('user:1')}")
print(f"session:abc → {cache.get('session:abc')}")

print("等待 2 秒...")
time.sleep(2)
print(f"session:abc (过期后) → {cache.get('session:abc')}")
print(f"user:1 (未过期) → {cache.get('user:1')}")

cache.delete("user:1")
print(f"user:1 (删除后) → {cache.get('user:1')}")

### 惰性删除 vs 主动删除

`MemoryCache` 采用的是 **惰性删除**：过期的 key 不会被后台定时清理，而是在 `get()` 时检查到过期才返回 `None`。

- **优点**：实现简单，不需要额外的清理线程
- **缺点**：过期的 key 仍然占用内存，直到被访问时才会被发现
- **Redis 的做法**：结合惰性删除 + 定期采样清理（折中方案）

---
## 六、工厂模式 + 依赖注入

项目通过环境变量决定使用哪种缓存后端：

```python
def create_cache() -> CacheBackend:
    backend = os.getenv("CACHE_BACKEND", "memory")
    if backend == "redis":
        return RedisCache()
    return MemoryCache()
```

这是一个典型的 **工厂模式**：调用者不关心具体实现，由工厂函数根据配置创建实例。

在 FastAPI 中通过 **生成器依赖注入** 使用：

```python
_cache: CacheBackend | None = None

def get_cache() -> Generator[CacheBackend, Any, None]:
    global _cache
    if _cache is None:
        _cache = create_cache()  # 单例：只创建一次
    yield _cache                 # yield 让 FastAPI 管理生命周期
```

在路由中这样使用：

```python
@router.get("/download/{album_id}")
def download(album_id: int, cache: CacheBackend = Depends(get_cache)):
    ...
```

In [ ]:
from collections.abc import Generator
from typing import Any

# 模拟工厂模式
class RedisCache:  # 简化版，实际项目中会连接真正的 Redis
    def get(self, key: str) -> str | None:
        return f"[Redis] {key}"
    def set(self, key: str, value: str, ex: int | None = None) -> None:
        pass
    def delete(self, key: str) -> None:
        pass
    def exists(self, key: str) -> bool:
        return True

def create_cache(backend: str = "memory") -> CacheBackend:
    if backend == "redis":
        return RedisCache()
    return MemoryCache()

# 通过配置切换后端，调用代码完全不变
mem_cache = create_cache("memory")
redis_cache = create_cache("redis")

mem_cache.set("test", "hello")
print(f"MemoryCache: {mem_cache.get('test')}")
print(f"RedisCache:  {redis_cache.get('test')}")

# 单例模式（项目中的 get_cache 做的事）
_cache_instance = None

def get_cache() -> Generator:
    global _cache_instance
    if _cache_instance is None:
        _cache_instance = create_cache("memory")
    yield _cache_instance

# 每次调用 get_cache() 都返回同一个实例
cache1 = next(get_cache())
cache2 = next(get_cache())
print(f"\n是否同一个实例: {cache1 is cache2}")

---
## 七、Double-Check Locking 实战 —— 项目中的真实用法

看项目 `app/services/comic.py` 中的 `download_and_merge_pdf` 函数：

```python
def download_and_merge_pdf(album_id: int, cache: CacheBackend) -> PdfFileResponse:
    cache_key = f"jmcomic:{album_id}"

    # ① 快速路径：缓存命中且文件存在，无需加锁
    cached_path = cache.get(cache_key)
    if cached_path is not None:
        pdf_path = Path(cached_path)
        if pdf_path.exists():
            return PdfFileResponse(pdf_path, filename=pdf_path.name)
        else:
            cache.delete(cache_key)

    # ② 对同一 album_id 加锁，防止并发重复下载
    with _album_lock.acquire(album_id):
        # ③ double-check：拿到锁后再查一次缓存
        cached_path = cache.get(cache_key)
        if cached_path is not None:
            pdf_path = Path(cached_path)
            if pdf_path.exists():
                return PdfFileResponse(pdf_path, filename=pdf_path.name)
            else:
                cache.delete(cache_key)

        # ④ 真正执行下载
        download_album(album_id, option)
        pdf_path = DOWNLOAD_DIR / f"{album_id}.pdf"
        cache.set(cache_key, str(pdf_path))

    return PdfFileResponse(pdf_path, filename=pdf_path.name)
```

### 流程图

```
请求进来
  │
  ├─ 缓存命中？── 是 ──→ 直接返回（快速路径，不加锁）
  │
  └─ 缓存未命中
       │
       ├─ 获取 KeyLock(album_id)
       │     │
       │     ├─ 再次查缓存（double-check）── 命中 ──→ 返回
       │     │
       │     └─ 未命中 → 执行下载 → 写入缓存
       │
       └─ 释放锁，返回结果
```

### 为什么要 double-check？

假设 3 个请求同时请求同一个 `album_id`：

1. 三个请求都通过 ① 快速路径，缓存未命中
2. 三个请求都尝试获取锁，只有**请求 A** 拿到锁
3. 请求 B、C 排队等待
4. 请求 A 下载完成，写入缓存，释放锁
5. 请求 B 拿到锁 → **double-check 发现缓存已有** → 直接返回，不重复下载
6. 请求 C 同理

如果没有 double-check，B 和 C 都会重复下载！

In [ ]:
# 用简化的代码模拟 double-check locking 的效果

class FakeCache:
    def __init__(self):
        self._data = {}
    def get(self, key):
        return self._data.get(key)
    def set(self, key, value):
        self._data[key] = value

cache = FakeCache()
key_lock = KeyLock()
download_count = 0  # 记录实际下载次数
log3 = []

def request_download(request_id: str, resource_id: int):
    cache_key = f"res:{resource_id}"

    # ① 快速路径
    cached = cache.get(cache_key)
    if cached is not None:
        log3.append(f"{request_id}: 快速路径命中！")
        return cached

    # ② 加锁
    with key_lock.acquire(resource_id):
        # ③ double-check
        cached = cache.get(cache_key)
        if cached is not None:
            log3.append(f"{request_id}: double-check 命中！避免重复下载")
            return cached

        # ④ 模拟耗时下载
        global download_count
        download_count += 1
        log3.append(f"{request_id}: 正在下载... (第 {download_count} 次实际下载)")
        time.sleep(0.5)
        result = f"data_of_{resource_id}"
        cache.set(cache_key, result)
        log3.append(f"{request_id}: 下载完成，已写入缓存")
        return result

# 5 个并发请求同时请求同一个 resource_id=42
threads = [
    threading.Thread(target=request_download, args=(f"Req-{i}", 42))
    for i in range(5)
]
for t in threads:
    t.start()
for t in threads:
    t.join()

for line in log3:
    print(line)
print(f"\n5 个请求，实际下载次数: {download_count}（如果没有 double-check 会是 5 次）")

---

## 总结

| 组件 | 解决的问题 | 关键技术 |
|------|-----------|----------|
| `RWLock` | 读多写少场景的并发性能 | `Condition` + 读写分离 + 写者优先 |
| `KeyLock` | 不同资源之间的并发隔离 | `dict[key → Lock]` + 惰性创建 |
| `CacheBackend` (Protocol) | 缓存实现的可替换性 | 结构化子类型（鸭子类型 + 类型检查）|
| `MemoryCache` | 本地线程安全缓存 | `RWLock` + 惰性过期 |
| `RedisCache` | 分布式缓存（生产环境）| 委托给 Redis 客户端 |
| `create_cache` | 按配置选择后端 | 工厂模式 |
| Double-Check Locking | 防止缓存击穿时的重复计算 | 快速路径 + 加锁 + 二次检查 |

### 核心设计原则

1. **面向接口编程**：依赖 `CacheBackend` 协议而非具体实现
2. **锁粒度最小化**：`KeyLock` 按资源 ID 分锁，`RWLock` 读写分离
3. **快速路径优先**：先不加锁查缓存，命中就直接返回
4. **Double-Check**：拿到锁后再确认一次，避免重复工作